# 编码转换：将分类特征变成机器学习能读懂的数字
> 难度标签：初级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：01_数据预处理 | 源文件：编码转换.py | 核心功能：6种分类特征编码策略的完整演示与对比

## 概述

现实世界的数据充满了"类别"——颜色、尺寸、品牌、学历、城市……但机器学习模型只认数字。**编码转换**就是把人类可理解的分类标签变成模型能处理的数值形式的过程。

听起来简单，但编码方式选错了，模型的效果可能天差地别。比如你把"红=0, 蓝=1, 绿=2"喂给线性回归，模型会天真地认为"绿 > 蓝 > 红"，这显然荒谬。

这个脚本演示了 **6 种主流编码方法**，从最朴素的标签编码到信息量最大的目标编码，每种方法都有其适用场景和陷阱。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

## 数学原理

### 1. 标签编码的虚假序关系

**代码对应**：`LabelEncoder().fit_transform(data["颜色"])` 将类别映射为整数。

标签编码定义映射 $f: \mathcal{C} \to \{0, 1, 2, \ldots, K-1\}$，其中 $\mathcal{C}$ 为类别集合。映射按字母序确定：

$$f(c_i) = \text{rank}(c_i \text{ in sorted } \mathcal{C})$$

**问题**：线性模型假设输入之间存在线性关系。如果颜色编码为 $\text{红}=0, \text{蓝}=1, \text{绿}=2$，线性回归会学到：

$$y = w_0 \cdot 0 + w_1 \cdot 1 + w_2 \cdot 2$$

隐含假设"绿色的权重是蓝色的两倍"——这在无序分类中毫无意义。

**正确用途**：标签编码只适用于两类场景：
- **目标变量** $y$ 的编码（模型输出不需要保持序关系）
- **树模型**的输入（树模型只比较大小关系来选择分裂点，不假设线性关系）

### 2. 独热编码与虚拟变量陷阱

**代码对应**：`OneHotEncoder(sparse_output=False).fit_transform(data[["颜色"]])`。

独热编码将类别 $c_k$ 映射为 $K$ 维标准基向量：

$$\mathbf{e}_k = (0, \ldots, 0, \underbrace{1}_{k\text{-th}}, 0, \ldots, 0) \in \{0,1\}^K$$

**虚拟变量陷阱**：设 $K$ 个独热列为 $d_1, d_2, \ldots, d_K$，则：

$$\sum_{k=1}^{K} d_k = 1 \quad \forall \text{ 样本}$$

这意味着 $d_K = 1 - \sum_{k=1}^{K-1} d_k$，第 $K$ 列是前 $K-1$ 列的线性组合。在线性回归中，设计矩阵 $\mathbf{X}$ 的列线性相关，导致：

$$\mathbf{X}^T\mathbf{X} \text{ 奇异，不可逆}$$

**代码对应**：`OneHotEncoder(drop="first")` 去掉第一列，使用 $K-1$ 个虚拟变量。此时：

$$\hat{y} = \beta_0 + \sum_{k=2}^{K} \beta_k d_k$$

$\beta_0$ 表示参考类别（被去掉的那个）的效应，$\beta_k$ 表示类别 $k$ 相对于参考类别的差异。

**注意**：使用正则化的模型（Ridge、Lasso）**不需要** `drop="first"`，因为正则化项打破了线性依赖。

### 3. 频率编码的信息论分析

**代码对应**：`data["品牌"].value_counts(normalize=True).to_dict()` 计算每个类别的频率。

频率编码：$f(c) = \frac{n_c}{N}$，其中 $n_c$ 为类别 $c$ 的样本数，$N$ 为总样本数。

**频率编码的信息损失**：假设两个不同类别 $c_1 \neq c_2$ 的频率恰好相同，即 $f(c_1) = f(c_2)$，则它们被映射为相同的数值——编码后的模型无法区分这两个类别。类别数越多，这种碰撞概率越高。

从信息论角度，频率编码将 $\log_2 K$ 比特的类别信息压缩为一个实数。如果类别分布不均匀（少数类别频率极低），信息损失更严重。

### 4. 目标编码与贝叶斯收缩

**代码对应**：`data.groupby("颜色")["价格"].mean().to_dict()` 计算每个类别的目标均值。

朴素目标编码：$\hat{\theta}_c = \bar{y}_c = \frac{1}{n_c}\sum_{i: x_i = c} y_i$

**过拟合风险**：当 $n_c$ 很小时，$\bar{y}_c$ 极不稳定。极端情况 $n_c = 1$ 时，编码值就是 $y$ 本身——完美的信息泄露。

**贝叶斯收缩（Bayesian Target Encoding）**引入正则化：

$$\hat{\theta}_c^{\text{Bayes}} = \frac{n_c \bar{y}_c + m \bar{y}}{n_c + m}$$

其中 $\bar{y}$ 为全局均值，$m$ 为平滑参数（先验的"等效样本数"）。

**数学推导**：假设类别 $c$ 的目标变量服从 $y|c \sim \mathcal{N}(\theta_c, \sigma^2)$，先验 $\theta_c \sim \mathcal{N}(\bar{y}, \tau^2)$，则后验均值为：

$$\mathbb{E}[\theta_c | \text{data}] = \frac{n_c/\sigma^2}{n_c/\sigma^2 + 1/\tau^2} \bar{y}_c + \frac{1/\tau^2}{n_c/\sigma^2 + 1/\tau^2} \bar{y}$$

令 $m = \sigma^2 / \tau^2$，即得上面的贝叶斯收缩公式。

**直觉**：当 $n_c$ 大时，$\hat{\theta}_c^{\text{Bayes}} \approx \bar{y}_c$（数据主导）；当 $n_c$ 小时，$\hat{\theta}_c^{\text{Bayes}} \approx \bar{y}$（先验主导）。平滑参数 $m$ 控制收缩强度。

### 5. 编码维度与稀疏性的权衡

各种编码方法的维度变化：

| 编码方法 | 原始 1 列 $\to$ 编码后 | 稀疏性 |
|----------|----------------------|--------|
| 标签编码 | 1 列 | 无 |
| 独热编码 | $K$ 列 | 极高（每行仅 1 个 1） |
| 独热+drop | $K-1$ 列 | 高 |
| 频率编码 | 1 列 | 无 |
| 目标编码 | 1 列 | 无 |

当类别数 $K$ 很大（如 $K = 10000$ 的城市编码），独热编码会将特征空间从 1 维膨胀到 10000 维——这就是**维度灾难**的直接来源。此时应使用频率编码、目标编码或 Embedding。

### 6. OneHotEncoder 的矩阵表示

**代码对应**：`OneHotEncoder` 生成的编码矩阵是**稀疏矩阵**。

设 $n$ 个样本的类别向量为 $\mathbf{c} = (c_1, \ldots, c_n)^T$，独热编码矩阵 $\mathbf{H} \in \{0,1\}^{n \times K}$ 满足：

$$H_{ik} = \begin{cases} 1 & \text{if } c_i = k \\ 0 & \text{otherwise} \end{cases}$$

矩阵 $\mathbf{H}$ 的秩为 $K-1$（因为各行之和为常数 1），这就是虚拟变量陷阱的矩阵解释。

稀疏度为 $1 - 1/K$，即当 $K = 100$ 时，99% 的元素为 0。sklearn 使用稀疏矩阵存储可以大幅节省内存。

## 代码结构

| 变量/模块 | 职责 |
|-----------|------|
| data | 包含颜色、尺寸、品牌、价格的示例 DataFrame |
| LabelEncoder | sklearn 标签编码器，将类别映射为整数 |
| OrdinalEncoder | sklearn 有序编码器，支持自定义顺序 |
| OneHotEncoder | sklearn 独热编码器，生成 0/1 矩阵 |
| pd.get_dummies | pandas 快捷独热编码 |
| 
req_map / 	arget_mean | 手动实现的频率编码和目标编码 |

脚本按 **编码复杂度递增** 的顺序组织：标签编码 → 有序编码 → 独热编码 → 频率编码 → 目标编码，最后给出对比总结。

### 1. 构造示例数据

构造包含特定特征的数据，用于测试算法的鲁棒性。

In [ ]:
data = pd.DataFrame({
    "颜色": ["红", "蓝", "绿", "红", "蓝", "绿", "红", "蓝", "绿", "红"],
    "尺寸": ["S", "M", "L", "XL", "M", "S", "L", "XL", "M", "S"],
    "品牌": ["A", "B", "A", "C", "B", "A", "C", "B", "A", "C"],
    "价格": [100, 200, 150, 300, 250, 120, 180, 280, 160, 220],
# --- 继续 ---
})
print("=== 原始数据 ===")
print(data)

### 2. 标签编码（LabelEncoder）

运行 2. 标签编码（LabelEncoder） 的代码，观察算法在该环节的行为。

In [ ]:
# 将每个类别映射为一个整数，适用于有序分类或目标变量
le = LabelEncoder()
data["颜色_标签"] = le.fit_transform(data["颜色"])
print(f"\n=== 标签编码 ===")
print(f"映射关系: {dict(zip(le.classes_, le.transform(le.classes_)))}")
print(data[["颜色", "颜色_标签"]])

# 注意：标签编码会引入数值大小关系（0 < 1 < 2），对无序分类可能不合适

### 3. 有序编码（OrdinalEncoder）

运行 3. 有序编码（OrdinalEncoder） 的代码，观察算法在该环节的行为。

In [ ]:
# 适用于有明确顺序的分类特征
# 注意：需要手动定义顺序，否则按字母序排列
size_order = [["S", "M", "L", "XL"]]
oe = OrdinalEncoder(categories=size_order)
data["尺寸_有序"] = oe.fit_transform(data[["尺寸"]])
print(f"\n=== 有序编码 ===")
print(f"顺序: S=0, M=1, L=2, XL=3")
# --- 输出结果 ---
print(data[["尺寸", "尺寸_有序"]])

### 4. 独热编码（OneHotEncoder）

运行 4. 独热编码（OneHotEncoder） 的代码，观察算法在该环节的行为。

In [ ]:
# 将每个类别转为独立的 0/1 列，适用于无序分类特征
ohe = OneHotEncoder(sparse_output=False, drop=None)  # drop="first" 可避免多重共线性
color_encoded = ohe.fit_transform(data[["颜色"]])
color_cols = ohe.get_feature_names_out(["颜色"])
data_ohe = pd.DataFrame(color_encoded, columns=color_cols)
print(f"\n=== 独热编码 ===")
# --- 输出结果 ---
print(f"类别: {ohe.categories_[0]}")
print(data_ohe)

# drop="first"：去掉第一列，避免虚拟变量陷阱（多重共线性）
ohe_drop = OneHotEncoder(sparse_output=False, drop="first")
color_drop = ohe_drop.fit_transform(data[["颜色"]])
color_drop_cols = ohe_drop.get_feature_names_out(["颜色"])
print(f"\n=== 独热编码（drop='first'）===")
print(pd.DataFrame(color_drop, columns=color_drop_cols))

### 5. Pandas get_dummies

运行 5. Pandas get_dummies 的代码，观察算法在该环节的行为。

In [ ]:
# 更方便的一键独热编码方式
data_dummies = pd.get_dummies(data[["颜色", "品牌"]], prefix=["颜色", "品牌"])
print(f"\n=== pd.get_dummies ===")
print(data_dummies)

### 6. 频率编码

运行 6. 频率编码 的代码，观察算法在该环节的行为。

In [ ]:
# 用类别出现的频率替代类别值，适用于高基数特征
freq_map = data["品牌"].value_counts(normalize=True).to_dict()
data["品牌_频率"] = data["品牌"].map(freq_map)
print(f"\n=== 频率编码 ===")
print(f"频率映射: {freq_map}")
print(data[["品牌", "品牌_频率"]])

### 7. 目标编码（均值编码）

运行 7. 目标编码（均值编码） 的代码，观察算法在该环节的行为。

In [ ]:
# 用目标变量的条件均值替代类别值，注意：需防止过拟合（通常在训练集上计算，应用到测试集）
# 此处仅作演示，实际应先划分训练/测试集
target_mean = data.groupby("颜色")["价格"].mean().to_dict()
data["颜色_目标编码"] = data["颜色"].map(target_mean)
print(f"\n=== 目标编码 ===")
print(f"目标均值映射: {target_mean}")
print(data[["颜色", "价格", "颜色_目标编码"]])

### 8. 对比总结

对比不同模型或配置的性能差异。

In [ ]:
print("\n=== 编码策略对比 ===")
print("1. 标签编码: 适合有序分类或目标变量，简单但引入虚假顺序")
print("2. 有序编码: 适合有明确顺序的特征（如 S<M<L<XL）")
print("3. 独热编码: 适合无序分类，维度随类别数增长，注意多重共线性")
print("4. 频率编码: 适合高基数特征，保留频率信息但丢失类别语义")
# --- 输出结果 ---
print("5. 目标编码: 信息量大但易过拟合，需交叉验证或正则化")

## 关键代码解释

### 1. 标签编码（LabelEncoder）—— 最简单但最危险

```python
le = LabelEncoder()
data["颜色_标签"] = le.fit_transform(data["颜色"])
```


it 阶段学习所有唯一类别，	ransform 阶段按字母序分配整数（红→0, 蓝→0, 绿→1 取决于中文排序）。**核心陷阱**：标签编码引入了虚假的大小关系。对于线性模型，这意味着"绿色的权重是蓝色的两倍"——完全是无稽之谈。

**正确用途**：目标变量（y）的编码、树模型（树模型只看分裂点，不看数值大小）。

### 2. 有序编码（OrdinalEncoder）—— 给类别排座次

```python
size_order = [["S", "M", "L", "XL"]]
oe = OrdinalEncoder(categories=size_order)
data["尺寸_有序"] = oe.fit_transform(data[["尺寸"]])
```

关键在于 categories 参数——你手动定义了 S < M < L < XL 的顺序。如果不传这个参数，OrdinalEncoder 会按字母序排列，可能得到错误的顺序。

**适用场景**：教育程度（高中 < 本科 < 硕士 < 卢布）、评价等级（差 < 中 < 好 < 优秀）等天然有序的特征。

### 3. 独热编码（OneHotEncoder）—— 安全但膨胀

```python
ohe = OneHotEncoder(sparse_output=False, drop=None)
color_encoded = ohe.fit_transform(data[["颜色"]])
```

每个类别变成一列 0/1 向量。drop="first" 参数去掉第一列以避免**虚拟变量陷阱**（多重共线性）——当所有列加起来恒等于 1 时，线性回归的设计矩阵变得奇异。

```python
ohe_drop = OneHotEncoder(sparse_output=False, drop="first")
```

**代价**：如果一个特征有 1000 个类别，独热编码会生成 999 列（或 1000 列）。这就是"维度灾难"的来源之一。

### 4. 频率编码 —— 用出现次数说话

```python
freq_map = data["品牌"].value_counts(normalize=True).to_dict()
data["品牌_频率"] = data["品牌"].map(freq_map)
```

简单粗暴：用类别出现的频率（占比）替代类别值。**优势**：不增加维度，天然适合高基数特征（比如城市、商品 ID）。**劣势**：不同类别可能有相同频率，导致信息丢失。

### 5. 目标编码（Target Encoding）—— 信息量最大但最危险

```python
target_mean = data.groupby("颜色")["价格"].mean().to_dict()
data["颜色_目标编码"] = data["颜色"].map(target_mean)
```

用目标变量的条件均值替代类别值。信息量巨大，但**极其容易过拟合**——如果某个类别只有一个样本，编码值就是目标变量本身。实际使用时必须：
- 在训练集上计算，应用到测试集
- 使用交叉验证（K-Fold Target Encoding）
- 或者加入贝叶斯平滑（Bayesian Target Encoding）

## 注意事项

1. **先分后编**：所有编码器必须在训练集上 
it，然后分别 	ransform 训练集和测试集，否则会造成数据泄露
2. **树模型 vs 线性模型**：树模型对编码方式不敏感（XGBoost 甚至支持原生分类特征），线性模型则需要谨慎选择
3. **高基数陷阱**：独热编码对高基数特征（如用户 ID）会产生灾难性的维度膨胀，此时应考虑频率编码、目标编码或 Embedding
4. **缺失值处理**：scikit-learn 的编码器默认不处理缺失值，需要先填充或使用 handle_unknown="ignore" 参数
5. **虚拟变量陷阱**：线性回归中使用独热编码时，记得 drop="first" 或使用正则化

## 总结与延伸

以上代码展示了 **编码转换** 的完整流程。

**解读要点**：
- 观察处理前后数据的 **统计分布变化**
- 关注缺失值填充策略对下游模型的影响
- 对比不同缩放方法的适用场景

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **Binary Encoding**：先用标签编码转整数，再把整数转为二进制位，比独热编码更紧凑，适合中等基数特征
- **Target Encoding 的高级变体**：James-Stein 编码、M-估计编码、Leave-One-Out 编码，都是为了解决过拟合问题
- **Embedding**：深度学习中的"终极编码方案"——把类别映射到低维稠密向量空间，既保留语义又控制维度。推荐关注 category_encoders 库，它提供了 15+ 种编码方式的一致 API
